# Compute horizontal ACC maps

Computes and caches spatial anomaly-correlation maps for configurable lead windows.


In [ ]:
from pathlib import Path
INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_out")

groups = ("Jan2012",)
runs = ("fc",)
selected = {"ERA5": ("TREFHT",)}
lead_bins = ((1,7), (8,14), (15,28), (29,42), (43,56))
lead_labels = ("wk1", "wk2", "wk3-4", "wk5-6", "wk7-8")
force_compute = False
compute_member_stats = True
save_member_maps = False
verbose = True


In [ ]:
from pathlib import Path
import sys

repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))

from configs.output_paths import workflow_output_dirs
from configs.s2s_config import EXPERIMENT_GROUPS, OBS_VARIABLES, PCC_REGIONS, TCC_REGIONS, LEAD_WINDOWS, WEEKLY_WINDOWS, AVAILABLE_PCC_METRICS, AVAILABLE_PCC_MAP_METRICS, AVAILABLE_TCC_METRICS
from util.s2s_experiments import build_experiments
from util.s2s_observations import OBS_REGISTRY, get_obs_file


## Notes: available selections

- Experiment groups: `Jan2012`, `Jun2012`.
- Runs: `fc` (standard forecast) or `wc` (water-cycle forecast).
- Observations and variables: `ERA5` supports UBOT, VBOT, PRECT, PSL, T925, Q925, U850, V850, T850, Q850, Z500, OMEGA850/500/200, U/V500, U/V200, FLDS/C, FLNS/C, FSDS/C, FSNS/C, LHFLX, SHFLX, QFLX, TREFHT, TS, PRECSL, PRECC, and PRECL; `GPCP` and `GPM` support PRECT; `NOAA-OLR` supports FLUT.
- Regions: GLOBAL, land/ocean where applicable, NINO3.4, MAR_CONT, CONUS, NH_POLAR.
- Existing NetCDF diagnostics are reused. Set `force_compute = True` to rebuild selected products.

Metrics: ensemble-mean ACC maps, sample counts, optional member mean/quantiles and member maps.

Exact cached fields: `ACC_EMEAN`, `N_SAMPLES_EMEAN`; with member statistics, `ACC_MEMBER_MEAN`, `ACC_MEMBER_STD`, `P_ACC_POS`, `N_SAMPLES_MEMBER_MEAN`, `N_ENS_USED`; with full member maps, `ACC_MEMBER`, `N_SAMPLES_MEMBER`.


In [ ]:
from util.s2s_pcc_metrics import FileCollectionConfig, S2SFileCollector, BiweeklyAccMapConfig, BiweeklyAccMapAssessor

data_dir, figure_dir = workflow_output_dirs("fcst_s2s_pcc", "acc_maps", output_root=DIAGNOSTIC_OUTPUT_ROOT)
exp_dict = build_experiments(str(INPUT_DATA_ROOT))
for group in groups:
    meta = EXPERIMENT_GROUPS[group]
    for run in runs:
        for obs, variables in selected.items():
            var_map = {v: OBS_VARIABLES[obs][v] for v in variables}
            cfg = FileCollectionConfig(group=group, freq=meta["freq"], run=run, obs=obs, period=meta["period"], model_template="%(var)s.%(ens)s.%(year)d.nc")
            collector = S2SFileCollector(exp_list=EXPERIMENT_GROUPS, exp_dict=exp_dict, obs_registry=OBS_REGISTRY, s2s_var_dict=var_map, get_obs_file_func=get_obs_file)
            files = collector.collect(cfg, vars_to_process=list(variables), verbose=verbose)
            assessor = BiweeklyAccMapAssessor(BiweeklyAccMapConfig(init_time=meta["init_date"], lead_bins=lead_bins, lead_bin_labels=lead_labels, overwrite=force_compute, compute_member_stats=compute_member_stats, save_member_maps=save_member_maps, verbose=verbose))
            for var in variables:
                if var not in files: continue
                expected = [data_dir / f"s2s_accmap_{group}_{meta['freq']}_{run}_{obs}_{exp}_{var}.nc" for exp in meta["models"]]
                if expected and all(p.exists() for p in expected) and not force_compute:
                    print(f"[REUSE] {var}: {len(expected)} map files"); continue
                ds = assessor.compute({var: files[var]}, period=meta["period"], obs_var_override=var_map)
                print(assessor.save_per_var_exp(ds, out_dir=str(data_dir), group=group, freq=meta["freq"], run=run, obs=obs, overwrite=force_compute))
